In [ ]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

import os
os.environ["HF_HOME"] = "/shared/data3/pk36/.cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"

import argparse
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from dataclasses import dataclass
import json_repair
import re
import json
from typing import List, Dict
from tqdm import tqdm
from collections import defaultdict
import time

from search import search_semantic_scholar, collect_snippets
from prompts import (
    create_initial_decomposition_prompt,
    create_target_domain_analysis_prompt,
    create_cross_domain_query_prompt,
    create_cross_domain_analysis_prompt,
    initial_decomposition_schema,
    target_domain_analysis_schema,
    cross_domain_queries_schema,
    cross_domain_analysis_schema
)
from classes import ResearchProblem, Question, Domain
from utils import prepare_output
from old.main import batch_llm_inference, retrieve_papers_for_question

/home/pk36/structured_survey/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-25 00:15:08 [__init__.py:216] Automatically detected platform cuda.


## Setup

In [2]:
@dataclass
class Args:
    problem_file: str = "data/robotics.txt"
    target_domain: str = "Computer Science"
    model_name: str = "Qwen/Qwen3-14B"
    output_dir: str = "output_debug"
    max_papers_per_query: int = 20

args = Args()

In [3]:
# Read problem statement
if os.path.exists(args.problem_file):
    with open(args.problem_file, "r") as f:
        problem_file_text = f.read()
        match = re.search(r"Problem Statement:\s*(.*)", problem_file_text)
        if match:
            problem_statement = match.group(1).strip()
        else:
            print("Could not find problem statement in file!")
    print(f"Problem Statement: {problem_statement}\n")
else:
    print(f"File {args.problem_file} does not exist!")

# Create output file path
output_file_name = os.path.splitext(os.path.basename(args.problem_file))[0] + f"_{args.max_papers_per_query}_results.json"
condensed_output_file_name = os.path.splitext(os.path.basename(args.problem_file))[0] + f"_{args.max_papers_per_query}_condensed.json"

args.output_file = os.path.join(args.output_dir, output_file_name)
args.condensed_output_file = os.path.join(args.output_dir, condensed_output_file_name)

# Create output directory if needed
os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

Problem Statement: Deep learning approaches, while effective for skill learning in robots, do not adequately address the complexities involved in teaching complete tasks that require understanding of complex logic and execution of related sub-tasks. This highlights a need for methodologies that enable robots to comprehend and remember sequences of actions based on single demonstrations to perform tasks accurately.



In [ ]:
# Initialize vLLM model
print("Loading model...")
llm = LLM(model=args.model_name, tensor_parallel_size=2)
print("Model loaded.\n")

Loading model...
INFO 01-25 00:15:42 [utils.py:233] non-default args: {'tensor_parallel_size': 2, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-14B'}
INFO 01-25 00:15:42 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 01-25 00:15:42 [model.py:1510] Using max model len 40960


2026-01-25 00:15:42,941	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-25 00:15:43 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=818663) INFO 01-25 00:15:43 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=818663) INFO 01-25 00:15:43 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-14B', speculative_config=None, tokenizer='Qwen/Qwen3-14B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=ObservabilityConfig(show_hidde

...

(EngineCore_DP0 pid=818663) (Worker_TP0 pid=818685) INFO 01-25 00:15:54 [default_loader.py:267] Loading weights took 4.52 seconds
(EngineCore_DP0 pid=818663) (Worker_TP1 pid=818687) INFO 01-25 00:15:55 [default_loader.py:267] Loading weights took 4.82 seconds
(EngineCore_DP0 pid=818663) (Worker_TP0 pid=818685) INFO 01-25 00:15:55 [gpu_model_runner.py:2653] Model loading took 13.8818 GiB and 5.161276 seconds
(EngineCore_DP0 pid=818663) (Worker_TP1 pid=818687) INFO 01-25 00:15:55 [gpu_model_runner.py:2653] Model loading took 13.8818 GiB and 5.489775 seconds
(EngineCore_DP0 pid=818663) (Worker_TP0 pid=818685) INFO 01-25 00:16:04 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/69891fa5b4/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=818663) (Worker_TP1 pid=818687) INFO 01-25 00:16:04 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/69891fa5b4/rank_1_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pi

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:05<00:00, 11.59it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 12.94it/s]


(EngineCore_DP0 pid=818663) (Worker_TP0 pid=818685) INFO 01-25 00:16:22 [custom_all_reduce.py:203] Registering 8262 cuda graph addresses
(EngineCore_DP0 pid=818663) (Worker_TP1 pid=818687) INFO 01-25 00:16:22 [custom_all_reduce.py:203] Registering 8262 cuda graph addresses
(EngineCore_DP0 pid=818663) (Worker_TP0 pid=818685) INFO 01-25 00:16:23 [gpu_model_runner.py:3480] Graph capturing finished in 10 secs, took 0.97 GiB
(EngineCore_DP0 pid=818663) (Worker_TP1 pid=818687) INFO 01-25 00:16:23 [gpu_model_runner.py:3480] Graph capturing finished in 10 secs, took 0.97 GiB
(EngineCore_DP0 pid=818663) INFO 01-25 00:16:23 [core.py:210] init engine (profile, create kv cache, warmup model) took 27.70 seconds
INFO 01-25 00:16:24 [llm.py:306] Supported_tasks: ['generate']
Model loaded.



(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:26 [multiproc_executor.py:154] Worker proc VllmWorker-0 died unexpectedly, shutting down executor.
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710] EngineCore encountered a fatal error.
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710] Traceback (most recent call last):
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710]   File "/home/pk36/structured_survey/env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 701, in run_engine_core
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710]     engine_core.run_busy_loop()
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710]   File "/home/pk36/structured_survey/env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 726, in run_busy_loop
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710]     self._process_input_queue()
(EngineCore_DP0 pid=818663) ERROR 01-25 14:56:27 [core.py:710]   File "/home/pk36/structured_s

(EngineCore_DP0 pid=818663) Process EngineCore_DP0:
(EngineCore_DP0 pid=818663) Traceback (most recent call last):
(EngineCore_DP0 pid=818663)   File "/home/pk36/anaconda3/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=818663)     self.run()
(EngineCore_DP0 pid=818663)   File "/home/pk36/anaconda3/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=818663)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=818663)   File "/home/pk36/structured_survey/env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 712, in run_engine_core
(EngineCore_DP0 pid=818663)     raise e
(EngineCore_DP0 pid=818663)   File "/home/pk36/structured_survey/env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 701, in run_engine_core
(EngineCore_DP0 pid=818663)     engine_core.run_busy_loop()
(EngineCore_DP0 pid=818663)   File "/home/pk36/structured_survey/env/lib/python3.10/site-packages/vllm/v1/engine/core.py"

## Decomposition

In [37]:
prompt = create_initial_decomposition_prompt(problem_statement, args.target_domain)
messages = [{"role": "user", "content": prompt}]

decomposition_outputs = batch_llm_inference(
    llm, 
    [messages], 
    initial_decomposition_schema,
    temperature=0
)
decomposition_output = decomposition_outputs[0]

if decomposition_output is None:
    print("Failed to get decomposition output!")
else:
    print(json.dumps(decomposition_output, indent=2))

Processed prompts: 100%|██████████| 1/1 [00:16<00:00, 16.86s/it, est. speed input: 64.66 toks/s, output: 41.71 toks/s]

{
  "problem_statement": "Deep learning approaches, while effective for skill learning in robots, do not adequately address the complexities involved in teaching complete tasks that require understanding of complex logic and execution of related sub-tasks. This highlights a need for methodologies that enable robots to comprehend and remember sequences of actions based on single demonstrations to perform tasks accurately.",
  "target_domain": "Computer Science",
  "fine_grained_domain": "Robotics and Artificial Intelligence (AI) - Task and Motion Planning (TAMP)",
  "core_challenge": "Current deep learning methods for robotic task learning excel at low-level skill acquisition but fail to generalize across complex, multi-step tasks that require logical reasoning and hierarchical action sequencing. This is particularly challenging when learning from single demonstrations, as the system must infer both the high-level task structure and the low-level motor policies from sparse and ambiguous

In [38]:
# Create ResearchProblem object
research_problem = ResearchProblem.from_initial_decomposition(
    decomposition_output, 
    args.target_domain
)

print(f"Generated {len(research_problem.research_questions)} research questions:")
for q in research_problem.research_questions:
    print(f"  - {q.id}:\n\t\t-{q.domain_specific_question}\n\t\t-{q.domain_agnostic_question}")
print()

Generated 3 research questions:
  - q1:
		-How can task and motion planning systems infer hierarchical task structures from single, high-level demonstrations without explicit task decomposition?
		-How can systems infer nested structures of actions from limited observational data without prior decomposition?
  - q2:
		-What are the representational limitations of deep learning models in encoding sequential action plans that require logical consistency and temporal coherence?
		-What are the limitations of current models in encoding sequences of actions that require logical consistency and temporal alignment?
  - q3:
		-How can planning algorithms be integrated with deep learning models to enable robust execution of subtask sequences under uncertainty and partial observability?
		-How can planning algorithms be combined with learning models to execute sequences of subtasks under uncertainty and incomplete information?



## Target Domain Analysis

In [39]:
# Step 2a: Retrieve papers for all questions in target domain
print("\n2a. Retrieving papers from target domain...")
for question in research_problem.research_questions:
    print(f"  Retrieving for {question.id}...")
    papers = retrieve_papers_for_question(
        question, 
        research_problem.target_domain,
        max_papers=args.max_papers_per_query
    )
    research_problem.target_domain.add_question_papers(question, papers)
    print(f"    -Retrieved {len(papers)} papers")


2a. Retrieving papers from target domain...
  Retrieving for q1...
	 -Searching Semantic Scholar for query: hierarchical task learning from single demonstrations in domain: Computer Science.
	 -Searching Semantic Scholar for query: TAMP with sparse supervision in domain: Computer Science.
	 -Searching Semantic Scholar for query: task decomposition from single trajectories in domain: Computer Science.
	 -Searching Semantic Scholar for query: end-to-end TAMP without explicit decomposition in domain: Computer Science.
	 -Searching Semantic Scholar for query: single-shot task planning in robotics in domain: Computer Science.
    -Retrieved 20 papers
  Retrieving for q2...
	 -Searching Semantic Scholar for query: sequential action encoding in TAMP in domain: Computer Science.
	 -Searching Semantic Scholar for query: logical consistency in action sequences in domain: Computer Science.
	 -Searching Semantic Scholar for query: temporal coherence in deep planning models in domain: Computer Sci

In [40]:
# Step 2b: Batch analyze all questions in target domain
print("\n2b. Analyzing target domain papers (batch inference)...")

# Prepare batch of analysis prompts
analysis_messages_list = []
for question in research_problem.research_questions:
    papers = research_problem.target_domain.fetch_question_papers(question)
    
    if not papers:
        print(f"  Warning: No papers for {question.id}, skipping analysis")
        continue
    
    prompt = create_target_domain_analysis_prompt(
        research_problem=research_problem.problem_statement,
        domain_specific_question=question.domain_specific_question,
        domain_agnostic_question=question.domain_agnostic_question,
        question_rationale=question.rationale,
        papers_with_snippets=papers,
        target_domain=args.target_domain,
        fine_grained_domain=research_problem.fine_grained_domain
    )
    messages = [{"role": "user", "content": prompt}]
    analysis_messages_list.append(messages)

# Batch inference for all analyses
if analysis_messages_list:
    analysis_outputs = batch_llm_inference(
        llm,
        analysis_messages_list,
        target_domain_analysis_schema,
        temperature=0,  # Lower temperature for analysis,
        max_tokens=4096
    )


2b. Analyzing target domain papers (batch inference)...


Processed prompts: 100%|██████████| 3/3 [00:58<00:00, 19.65s/it, est. speed input: 411.17 toks/s, output: 100.28 toks/s]


In [41]:
# Process analysis results
for i, (question, analysis_output) in enumerate(zip(research_problem.research_questions, analysis_outputs)):
    if analysis_output is None:
        print(f"  Failed to analyze {question.id}")
        continue

    paper_relevance = {p["paper_title"]: p["is_relevant"] for p in analysis_output.get("paper_relevance", [])}
    paper_titles = list(research_problem.target_domain.fetch_question_papers(question).keys())
    
    question.target_domain_analysis = analysis_output
    research_problem.target_domain.add_question_analysis(question, analysis_output)
    # delete irrelevant papers (determined from analysis output)
    for paper in paper_titles:
        if paper in paper_relevance and not paper_relevance[paper]:
            research_problem.target_domain.del_question_paper(question, paper)
    
    # Determine if addressed
    assessment = analysis_output.get("overall_assessment", "largely unaddressed").lower()
    is_addressed = "substantially" in assessment or "partial" in assessment
    question.mark_as_addressed(is_addressed)
    
    print(f"  {question.id}: {assessment} ({question.domain_specific_question})")
    
    # Create sub-questions for remaining challenges
    remaining_challenges = analysis_output.get("remaining_challenges", [])
    for challenge_data in remaining_challenges:
        challenge = research_problem.add_remaining_challenge(question, challenge_data)
        print(f"    -> New challenge: {challenge.domain_specific_question}")

  q1: partially addressed (How can task and motion planning systems infer hierarchical task structures from single, high-level demonstrations without explicit task decomposition?)
    -> New challenge: How can TAMP systems robustly infer hierarchical task structures from single, high-level demonstrations without any prior decomposition or extensive training data?
  q2: partially addressed (What are the representational limitations of deep learning models in encoding sequential action plans that require logical consistency and temporal coherence?)
    -> New challenge: How can deep learning models be effectively trained to encode complex, long-horizon action sequences with both logical consistency and temporal coherence in TAMP?
  q3: partially addressed (How can planning algorithms be integrated with deep learning models to enable robust execution of subtask sequences under uncertainty and partial observability?)
    -> New challenge: How can planning algorithms be integrated with deep

In [42]:
for i, (question, analysis_output) in enumerate(zip(research_problem.research_questions, analysis_outputs)):
    print(f"  {question.id}: {assessment} ({question.domain_specific_question})")
    remaining_challenges = question.remaining_challenges
    for challenge in remaining_challenges:
        print(f"\t-> New challenge: {challenge.domain_specific_question}")
        print(f"\t\t-> Rationale: {challenge.rationale}")

  q1: partially addressed (How can task and motion planning systems infer hierarchical task structures from single, high-level demonstrations without explicit task decomposition?)
	-> New challenge: How can TAMP systems robustly infer hierarchical task structures from single, high-level demonstrations without any prior decomposition or extensive training data?
		-> Rationale: This challenge is fundamental to the research question, as it directly addresses the feasibility of learning complex hierarchical task structures from single demonstrations, which is a core requirement for practical TAMP systems. Existing work has made progress on learning hierarchical structures from limited demonstrations, but the robustness and generalizability of these methods in the single-demonstration setting remain unproven. The challenge lies in ensuring that the inferred hierarchical structures are both accurate and reusable across different tasks and environments, which requires a deeper understanding o

## Cross-Domain Query Generation

In [43]:
# Get all questions needing cross-domain search
questions_needing_cross_domain = research_problem.get_questions_needing_cross_domain()

print(f"\nFound {len(questions_needing_cross_domain)} questions needing cross-domain search:")
for q in questions_needing_cross_domain:
    print(f"  - {q.id}: {q.domain_agnostic_question}")

if not questions_needing_cross_domain:
    print("\nAll questions addressed in target domain! No cross-domain search needed.")
else:
    # Step 3a: Generate cross-domain queries (batch)
    print("\n3a. Generating cross-domain queries (batch inference)...")
    
    cross_domain_messages_list = []
    for question in questions_needing_cross_domain:
        # Get target domain assessment if available
        target_assessment = None
        if question.parent_question and question.parent_question.target_domain_analysis:
            # This is a remaining challenge (includes )
            target_assessment = question.rationale
        elif question.target_domain_analysis:
            # This is an original question (iterate over all challenges in target domain analysis)
            target_assessment = ""
            for challenge in question.remaining_challenges:
                target_assessment += f"- {challenge.rationale}\n"
        
        prompt = create_cross_domain_query_prompt(
            problem_statement=research_problem.problem_statement,
            domain_specific_question=question.domain_specific_question,
            domain_agnostic_question=question.domain_agnostic_question,
            question_rationale=question.rationale,
            target_domain=args.target_domain,
            fine_grained_domain=research_problem.fine_grained_domain,
            target_domain_assessment=target_assessment
        )
        messages = [{"role": "user", "content": prompt}]
        cross_domain_messages_list.append(messages)
    
    # Batch inference for cross-domain queries
    cross_domain_outputs = batch_llm_inference(
        llm,
        cross_domain_messages_list,
        cross_domain_queries_schema,
        temperature=0
    )


Found 3 questions needing cross-domain search:
  - c1: How can systems reliably infer nested structures of actions from minimal observational data without any prior decomposition or extensive training?
  - c1: How can models be trained to encode sequences of actions that require both logical consistency and temporal alignment, especially over long time horizons?
  - c1: How can planning and learning systems be combined to dynamically adapt to new sequences of tasks and environments with incomplete information and high uncertainty?

3a. Generating cross-domain queries (batch inference)...


Processed prompts: 100%|██████████| 3/3 [00:07<00:00,  2.66s/it, est. speed input: 417.63 toks/s, output: 112.64 toks/s]


In [44]:
# Process cross-domain query results
cross_domain_analysis_prompts = []
cross_domain_analysis_keys = []
for question, cross_domain_output in tqdm(zip(questions_needing_cross_domain, cross_domain_outputs), total=len(questions_needing_cross_domain)):
    if cross_domain_output is None:
        print(f"  Failed to generate cross-domain queries for {question.id}")
        continue
    
    question.cross_domain_queries = cross_domain_output
    
    print(f"\n  {question.domain_agnostic_question}:")
    for domain_search in cross_domain_output.get("cross_domain_searches", []):
        domain_name = domain_search["domain"]
        queries = domain_search["queries"]
        
        # Get or create domain
        domain = research_problem.get_or_create_domain(domain_name)
        domain.add_question_queries(question, queries)
        question.add_external_domain(domain)
        
        print(f"    - {domain_name}: {len(queries)} queries")
        papers = retrieve_papers_for_question(
            question,
            domain,
            max_papers=args.max_papers_per_query
        )
        
        # Conduct cross-domain analysis on domain papers
        cross_domain_analysis_prompt = create_cross_domain_analysis_prompt(
            problem_statement=research_problem.problem_statement,
            domain_agnostic_question=question.domain_agnostic_question,
            question_challenge=question.rationale,
            source_domain=domain_name,
            papers_with_snippets=papers,
            target_domain=research_problem.target_domain,
            fine_grained_domain=research_problem.fine_grained_domain
        )
        cross_domain_analysis_messages = [{"role": "user", "content": cross_domain_analysis_prompt}]
        cross_domain_analysis_prompts.append(cross_domain_analysis_messages)
        cross_domain_analysis_keys.append((question, domain))
        
        domain.add_question_papers(question, papers)
        domain_search["retrieved_papers"] = papers
        print(f"      -Retrieved {len(papers)} papers")

  0%|          | 0/3 [00:00<?, ?it/s]


  How can systems reliably infer nested structures of actions from minimal observational data without any prior decomposition or extensive training?:
    - Psychology: 3 queries
	 -Searching Semantic Scholar for query: sequence learning from observation in domain: Psychology.
Rate limited. Retrying after 10 seconds...
	 -Searching Semantic Scholar for query: observational learning mechanisms in domain: Psychology.
	 -Searching Semantic Scholar for query: action sequence generalization in domain: Psychology.
      -Retrieved 9 papers
    - Linguistics: 3 queries
	 -Searching Semantic Scholar for query: structure inference from minimal input in domain: Linguistics.
Rate limited. Retrying after 5 seconds...
	 -Searching Semantic Scholar for query: syntax from sparse data in domain: Linguistics.
	 -Searching Semantic Scholar for query: semantic parsing with limited examples in domain: Linguistics.
      -Retrieved 14 papers
    - Biology: 3 queries
	 -Searching Semantic Scholar for query:

 33%|███▎      | 1/3 [00:33<01:06, 33.04s/it]

      -Retrieved 15 papers

  How can models be trained to encode sequences of actions that require both logical consistency and temporal alignment, especially over long time horizons?:
    - Psychology: 3 queries
	 -Searching Semantic Scholar for query: sequential memory encoding in domain: Psychology.
	 -Searching Semantic Scholar for query: temporal binding mechanisms in domain: Psychology.
	 -Searching Semantic Scholar for query: cognitive sequence learning in domain: Psychology.
      -Retrieved 13 papers
    - Linguistics: 3 queries
	 -Searching Semantic Scholar for query: syntactic sequence modeling in domain: Linguistics.
	 -Searching Semantic Scholar for query: temporal coherence in language in domain: Linguistics.
	 -Searching Semantic Scholar for query: discourse structure encoding in domain: Linguistics.
      -Retrieved 11 papers
    - Biology: 3 queries
	 -Searching Semantic Scholar for query: developmental sequence encoding in domain: Biology.
	 -Searching Semantic Schol

 67%|██████▋   | 2/3 [00:51<00:24, 24.69s/it]

      -Retrieved 13 papers

  How can planning and learning systems be combined to dynamically adapt to new sequences of tasks and environments with incomplete information and high uncertainty?:
    - Psychology: 4 queries
	 -Searching Semantic Scholar for query: adaptive decision making in domain: Psychology.
	 -Searching Semantic Scholar for query: cognitive flexibility in domain: Psychology.
	 -Searching Semantic Scholar for query: uncertainty adaptation in domain: Psychology.
	 -Searching Semantic Scholar for query: sequential learning in domain: Psychology.
      -Retrieved 20 papers
    - Biology: 4 queries
	 -Searching Semantic Scholar for query: adaptive behavior in domain: Biology.
	 -Searching Semantic Scholar for query: neural plasticity in domain: Biology.
	 -Searching Semantic Scholar for query: sequential adaptation in domain: Biology.
Rate limited. Retrying after 10 seconds...
	 -Searching Semantic Scholar for query: environmental uncertainty in domain: Biology.
      -R

100%|██████████| 3/3 [01:34<00:00, 31.58s/it]

      -Retrieved 19 papers


In [13]:
# Reconstruct prompts for cross-domain analysis based on updated prompt template
reconstructed_cross_domain_analysis_prompts = []
for (question, domain) in cross_domain_analysis_keys:
    papers = domain.fetch_question_papers(question)
    
    cross_domain_analysis_prompt = create_cross_domain_analysis_prompt(
        problem_statement=research_problem.problem_statement,
        domain_agnostic_question=question.domain_agnostic_question,
        question_challenge=question.rationale,
        source_domain=domain.domain_name,
        papers_with_snippets=papers,
        target_domain=research_problem.target_domain,
        fine_grained_domain=research_problem.fine_grained_domain
    )
    cross_domain_analysis_messages = [{"role": "user", "content": cross_domain_analysis_prompt}]
    reconstructed_cross_domain_analysis_prompts.append(cross_domain_analysis_messages)

In [45]:
# Step 3b: Batch cross-domain analyses
print("\n3b. Analyzing cross-domain papers (batch inference)...")
if cross_domain_analysis_prompts:
    cross_domain_analysis_outputs = batch_llm_inference(
        llm,
        cross_domain_analysis_prompts,
        cross_domain_analysis_schema,
        temperature=0,
        max_tokens=4096
    )
    
    # # Process cross-domain analysis results
    # for (question, domain), analysis_output in zip(cross_domain_analysis_keys, cross_domain_analysis_outputs):
    #     if analysis_output is None:
    #         print(f"  Failed to analyze cross-domain papers for question '{question.id}' in domain '{domain.domain_name}'")
    #         continue
        
    #     question.add_cross_domain_analysis(domain, analysis_output)
    #     print(f"  Analyzed cross-domain papers for question '{question.id}' in domain '{domain.domain_name}'")


3b. Analyzing cross-domain papers (batch inference)...


Processed prompts: 100%|██████████| 9/9 [01:43<00:00, 11.52s/it, est. speed input: 571.33 toks/s, output: 180.00 toks/s]


In [ ]:
cross_domain_analysis_outputs

[{'conceptual_challenge': 'How can systems reliably infer nested structures of actions from minimal observational data without any prior decomposition or extensive training?',
  'source_domain': 'Psychology',
  'target_domain': 'Domain(domain_name=Computer Science)',
  'fine_grained_domain': 'Robotics and Artificial Intelligence (AI) - Task and Motion Planning (TAMP)',
  'paper_relevance': [{'paper_title': 'Inferring an Observer’s Prediction Strategy in Sequence Learning Experiments',
    'directly_addresses_challenge': True,
    'relevance_explanation': 'This paper directly addresses the challenge by exploring how observers infer prediction strategies from limited observational data, which is analogous to how systems might infer nested action structures from minimal demonstrations. It investigates the limits of inferring complex models from sparse data, which is central to the challenge.'},
   {'paper_title': 'Learning Sequences: Their Existence, Effect, and Evolution',
    'directly_

/home/pk36/anaconda3/lib/python3.10/multiprocessing/resource_tracker.py:224: UserWarning: resource_tracker: There appear to be 2 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


In [46]:
options = []
questions2domains = defaultdict(dict)
questions2domains["research_problem"] = research_problem.problem_statement
questions2domains["domain"] = research_problem.target_domain.domain_name
questions2domains["fine_grained_domain"] = research_problem.fine_grained_domain

for idx, ((q, domain), out) in enumerate(zip(cross_domain_analysis_keys, cross_domain_analysis_outputs)):
    if_relevant = [p["paper_title"] if p["directly_addresses_challenge"] else None for p in out["paper_relevance"]]
    num_relevant = sum([1 if p is not None else 0 for p in if_relevant])
    prop_relevant = num_relevant/len(if_relevant)
    options.append((idx, q.domain_specific_question, out["source_domain"], f": {num_relevant}/{len(if_relevant)}", prop_relevant))

    # if q.domain_specific_question not in questions2domains:
    #     questions2domains[q.domain_specific_question] = []
    if (prop_relevant > 0.5) and (out["challenge_sufficiency_assessment"]["is_challenge_addressed"]):
        if q.domain_specific_question not in questions2domains:
            if q.parent_question is not None:
                questions2domains[q.domain_specific_question]["parent_question"] = q.parent_question.domain_specific_question
                target_paper_info = {p:snippets for p, snippets in research_problem.target_domain.fetch_question_papers(q.parent_question).items()}
                questions2domains[q.domain_specific_question]["target_domain_papers"] = target_paper_info

            questions2domains[q.domain_specific_question]["rationale"] = q.rationale

        paper_info = {p.lower():snippets for p, snippets in domain.fetch_question_papers(q).items()}
        questions2domains[q.domain_specific_question][out["source_domain"]] = {'papers': {p:paper_info[p.lower()] for p in if_relevant if ((p is not None) and (p.lower() in paper_info))}, 'takeaways': out["solution_takeaways"], "remaining_challenge": out["challenge_sufficiency_assessment"]}
ranked_options = sorted(options, key=lambda x: x[-1], reverse=True)
for o in ranked_options:
    print(o)

(4, 'How can deep learning models be effectively trained to encode complex, long-horizon action sequences with both logical consistency and temporal coherence in TAMP?', 'Linguistics', ': 9/11', 0.8181818181818182)
(6, 'How can planning algorithms be integrated with deep learning models to dynamically adapt to novel subtask sequences and environments with high uncertainty and partial observability?', 'Psychology', ': 16/20', 0.8)
(3, 'How can deep learning models be effectively trained to encode complex, long-horizon action sequences with both logical consistency and temporal coherence in TAMP?', 'Psychology', ': 7/12', 0.5833333333333334)
(7, 'How can planning algorithms be integrated with deep learning models to dynamically adapt to novel subtask sequences and environments with high uncertainty and partial observability?', 'Biology', ': 5/15', 0.3333333333333333)
(0, 'How can TAMP systems robustly infer hierarchical task structures from single, high-level demonstrations without any p

In [47]:
with open(f"output_debug/{os.path.splitext(os.path.basename(args.problem_file))[0]}_recommendations.json", "w") as f:
    json.dump(questions2domains, fp=f, indent=2)